## Importation des bibliothèques

In [ ]:
import pandas as pd
from great_tables import GT
import matplotlib.pyplot as plt
import geopandas as gpd
import requests
import io
import geopandas as gpd
from shapely.geometry import Point
import folium

## Importation de la base de données

In [ ]:
# Récupération des données via l'url accessible sur : https://www.data.gouv.fr/datasets/recensement-des-equipements-sportifs-espaces-et-sites-de-pratiques
# Chargement long

# data = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/ea4f5879-af40-4e3e-949d-812d6eeb5e02")

# Récupération directement depuis le dernier csv téléchargé au préalable et stocké sur le s3

data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")
data.head(10)

## Analyses descriptives

In [ ]:
total_equipements = data.shape[0]
print(total_equipements)

### Répartition par type d'équipements

In [ ]:
equip_types = (
    data['equip_type_famille']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

equip_types.head(10).plot(kind='bar')
plt.title("Top 10 des types d'équipements (%)")
plt.xlabel("Type d'équipement")
plt.ylabel("Part (%)")
plt.show()

In [ ]:
equip_dep = (
    data
    .groupby('dep_nom')
    .size()
    .sort_values(ascending=False)
)

equip_dep.head(10).plot(kind='bar')
plt.title("Top 10 départements les plus équipés")
plt.xlabel("Département")
plt.ylabel("Nombre d'équipements")
plt.show()

In [ ]:
equip_reg = (
    data
    .groupby('reg_nom')
    .size()
    .sort_values(ascending=False)
)

equip_reg.head(3).plot(kind='bar')
plt.title("Top 3 des régions les plus équipés")
plt.xlabel("Région")
plt.ylabel("Nombre d'équipements")
plt.show()

### Génération d'une carte qui répértorie le nombre d'équipement par famille d'équipement

In [ ]:
def plot_carte_nb_equipement_par_famille(famille=None):
    new_data = data.copy()
    if famille is not None:
        new_data = data[data['equip_type_famille'] == famille]
    equip_dep = (
    new_data
    .groupby('dep_nom')
    .size()
    .sort_values(ascending=False)
    )

    #1. Conversion de la Series en DataFrame
    df = equip_dep.reset_index()
    df.columns = ["departement", "valeur"]

    #2. Téléchargement du GeoJSON
    url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
    response = requests.get(url)
    gdf = gpd.read_file(io.BytesIO(response.content))

    #3. Fusion
    gdf = gdf.merge(df, left_on="nom", right_on="departement", how="left")

    #4. Carte 
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))

    gdf.plot(
        column="valeur",
        ax=ax,
        cmap="YlOrRd",
        legend=True,
        legend_kwds={
            "label": "Nombre d'équipements",
            "orientation": "vertical",
            "shrink": 0.6,
        },
        missing_kwds={
            "color": "#e0e0e0",
            "label": "Données manquantes"
        },
        edgecolor="white",
        linewidth=0.5,
        vmin=df["valeur"].min(),
        vmax=df["valeur"].max()
    )

    ax.set_title("Nombre d'équipements par département", fontsize=16, fontweight="bold", pad=15)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_carte_nb_equipement_par_famille()


## Zoom sur l'ensemble des équipements "nautiques"

In [ ]:
data_nautique = data[data['equip_type_famille'] == "Site d'activités aquatiques et nautiques"]


colonnes = ['equip_numero', "inst_numero", "inst_nom", "inst_adresse", "inst_cp", "dep_code", "reg_code", "dep_nom", "reg_nom", "lib_bdv", "equip_nom", "equip_type_name", "equip_coordonnees", "aps_name"]
data_nautique = data_nautique[colonnes].reset_index(drop=True).copy()
mots = ["surf", "ski", "voile", "foil", "wake"]
pattern = "|".join(mots)
data_nautique = data_nautique[data_nautique['aps_name'].str.contains(pattern, case=False, na=False)]
data_nautique['dep_code'] = data_nautique['dep_code'].astype(str)
data_nautique

### Création de la carte folium

In [ ]:
# Séparer lat/lon (si equip_coordonnees est du type "lat, lon")
data_nautique[['lat', 'lon']] = data_nautique['equip_coordonnees'].str.split(',', expand=True).astype(float)

data_nautique['lat'] = data_nautique['lat'].dropna()
data_nautique['lon'] = data_nautique['lon'].dropna()
data_nautique['equip_nom'] = data_nautique['equip_nom'].dropna()
# Créer le GeoDataFrame
geometry = [Point(lon, lat) for lat, lon in zip(data_nautique['lat'], data_nautique['lon'])]
gdf = gpd.GeoDataFrame(data_nautique, geometry=geometry, crs=2154)

In [ ]:
m = folium.Map(location=[46.6, 2.3], zoom_start=6)

for _, row in data_nautique.dropna(subset=['lat', 'lon']).iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['aps_name']
    ).add_to(m)

#m

In [ ]:
# Nombre d'équipements par département
tab_dep = (
    data_nautique
    .groupby('dep_nom')
    .size()
    .reset_index(name='nb_equipements')
)
tab_dep

In [ ]:
# Top 3
top3 = tab_dep.nlargest(3, 'nb_equipements')

# Bottom 3
bottom3 = tab_dep.nsmallest(3, 'nb_equipements')

# Combiner
tab_final = pd.concat([top3, bottom3])

# Ajouter une colonne pour distinguer
tab_final['categorie'] = ['Top 3'] * 3 + ['Les 3 moins bons'] * 3

tab_final = tab_final.sort_values(
    by=['categorie', 'nb_equipements'],
    ascending=[False, False]
)

tab_final = tab_final.reset_index(drop=True)

tab_final

### Visualisation de la table finale

In [ ]:
table = (
    GT(tab_final)
    .tab_header(
        title="Départements les plus et moins équipés d'équipements nautiques)",
        subtitle="Top 3 et Bottom 3 en nombre d'équipements"
    )
    .cols_label(
        dep_nom="Nom du département",
        nb_equipements="Nombre d'équipements",
        categorie="Catégorie"
    )
    .fmt_number(columns=['nb_equipements'], decimals=0)
    .data_color(
        columns=['nb_equipements'],
        palette=["lightblue", "darkblue"]
    )
)

table

In [ ]:
#Focus sur les DROM
drom_numero = ['971', '972', '973', '974', '976']

data_drom = data_nautique[data_nautique['dep_code'].isin(drom_numero)]

data_drom.head(5)

In [ ]:
drom_equip = (
    data_drom
    .groupby('dep_nom')
    .size()
    )

drom_equip.plot(kind='bar', color='skyblue')
plt.title("Nombre d'équipements nautiques par département d'outre-mer")
plt.xlabel("Département")
plt.ylabel("Nombre d'équipement")
plt.xticks(rotation=45, ha='right')